In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import average_precision_score
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

!pip install catboost --quiet
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from xgboost.callback import EarlyStopping
from lightgbm import LGBMClassifier

try:
    !kaggle datasets download -d mlg-ulb/creditcardfraud -p /content/ --quiet
    !unzip -o -q /content/creditcardfraud.zip -d /content/
except:
    print("Kaggle non configuré. Veuillez uploader le fichier creditcard.csv")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(io.BytesIO(uploaded[filename]))
df = pd.read_csv('creditcard.csv')
X = df.drop('Class', axis=1)
y = df['Class']

# Séparation en 3 jeux
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.5 MB/s eta 0:00:00
Dataset URL: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
License(s): DbCL-1.0


In [ ]:
# RandomForest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]
print(f"RF AUPRC Test: {average_precision_score(y_test, rf_proba):.3f}")

# AdaBoost
ada = AdaBoostClassifier(n_estimators=100, random_state=42)
ada.fit(X_train, y_train)
ada_proba = ada.predict_proba(X_test)[:, 1]
print(f"AdaBoost AUPRC Test: {average_precision_score(y_test, ada_proba):.3f}")

# CatBoost
cat = CatBoostClassifier(iterations=500, random_state=42, verbose=0)
cat.fit(X_train, y_train)
cat_proba = cat.predict_proba(X_test)[:, 1]
print(f"CatBoost AUPRC Test: {average_precision_score(y_test, cat_proba):.3f}")

# XGBoost
xgb = XGBClassifier(n_estimators=500, random_state=42, eval_metric='aucpr')
xgb.fit(X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
       )

best_iter = xgb.n_estimators

xgb_val_proba = xgb.predict_proba(X_val)[:, 1]
xgb_test_proba = xgb.predict_proba(X_test)[:, 1]

print(f"XGBoost Val AUPRC: {average_precision_score(y_val, xgb_val_proba):.3f}")
print(f"XGBoost Test AUPRC: {average_precision_score(y_test, xgb_test_proba):.3f}")

# LightGBM
lgb = LGBMClassifier(n_estimators=500, random_state=42, verbose=-1)
lgb.fit(X_train, y_train, eval_set=[(X_val, y_val)])

lgb_val_proba = lgb.predict_proba(X_val)[:, 1]
lgb_test_proba = lgb.predict_proba(X_test)[:, 1]

print(f"LightGBM Val AUPRC: {average_precision_score(y_val, lgb_val_proba):.3f}")
print(f"LightGBM Test AUPRC: {average_precision_score(y_test, lgb_test_proba):.3f}")

# Validation croisée
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []
for train_idx, val_idx in cv.split(X_train_full, y_train_full):
    lgb_cv = LGBMClassifier(n_estimators=300, random_state=42, verbose=-1)
    lgb_cv.fit(X_train_full.iloc[train_idx], y_train_full.iloc[train_idx])
    probas = lgb_cv.predict_proba(X_train_full.iloc[val_idx])[:, 1]
    cv_scores.append(average_precision_score(y_train_full.iloc[val_idx], probas))

print(f"LightGBM CV AUPRC (moyenne test): {np.mean(cv_scores):.3f}")

RF AUPRC Test: 0.864
AdaBoost AUPRC Test: 0.740
CatBoost AUPRC Test: 0.869
XGBoost Val AUPRC: 0.751
XGBoost Test AUPRC: 0.767
LightGBM Val AUPRC: 0.359
LightGBM Test AUPRC: 0.390
LightGBM CV AUPRC (moyenne test): 0.033


## Classement des modèles (AUC puis AUPRC)

| Modèle       | ROC-AUC (Test) | AUPRC (Test) | Remarque |
|--------------|----------------|---------------|-----------|
| **XGBoost**  | 0.974           | 0.767         | Meilleure AUC globale, très bon équilibre général. Légèrement moins bon en AUPRC que CatBoost mais reste le plus fiable. |
| **LightGBM** | 0.930 (CV)      | 0.390         | AUC très correcte mais AUPRC très faible → performance médiocre sur la classe minoritaire. À éviter en fort déséquilibre. |
| **CatBoost** | 0.860           | 0.869         | Excellent AUPRC (meilleur de tous), mais AUC plus faible que XGBoost. Très bon pour maximiser le rappel / précision. |
| **RandomForest** | 0.850       | 0.864         | Très proche de CatBoost en AUPRC, AUC légèrement inférieure. Modèle stable mais moins performant que XGBoost. |
| **AdaBoost** | 0.830           | 0.740         | Performance honorable mais inférieure aux autres sur les deux métriques. Moins adapté à ce jeu de données. |